# Functions and Null Safety

Write reusable Kotlin functions and handle values that may be missing.

Lesson 1 used expressions to choose messages. Now we will put that logic behind named functions so callers can request a result without repeating the implementation. We will also make missing data part of the function's type contract. These are language skills you will use when an Android screen receives optional profile or note data.

## Learning Goals

By the end of this lesson, you will be able to:

- Declare and call a Kotlin function with a return type, default argument, and named arguments.
- Handle nullable text with a check, safe call, or fallback instead of an unnecessary assertion.
- Test present, missing, and empty input and explain their different results.

## Why This Matters

An app often displays data it does not control. A user may not have entered a name, or a server may omit an optional field. Repeating checks across every screen makes the app harder to change and easier to break.

A function gives one behavior a clear name and a consistent contract. Kotlin's nullable types make possible absence visible before the program runs. Together, these features help you decide what a screen should show when information is missing. We will use ordinary text output to test that decision before adding Android UI code.

## Check Your Starting Point

In Lesson 1, an `if` expression supplied one of two messages. What value would `if (title == "") "Add a title" else title` produce when `title` is `""`? What would it produce for `"Travel"`? Write the two results and explain which branch supplies each value.

In [ ]:
Your response:
Write your explanation here.

<details>
<summary>Show answer</summary>

The results are `Add a title` and `Travel`. Empty text matches the condition, so the first branch supplies the fallback. Present text uses the `else` branch. Both branches produce a `String`; this expression does not yet handle missing data.

</details>

## Video Demonstration

Watch a display-name function turn optional text into a dependable message. Follow which operation runs for a present value and which path is taken for a missing value.

<video controls preload="metadata" width="800" aria-label="Functions and Null Safety demonstration">
<source src="media/02_functions_and_null_safety/lesson.mp4" type="video/mp4">
<track kind="captions" src="media/02_functions_and_null_safety/captions.vtt" srclang="en" label="English">
Your browser does not support embedded video.
</video>

[Read the video transcript and visual description](media/02_functions_and_null_safety/transcript.md).

## Concept

### Give a Behavior a Function

A **function** is a named operation that can receive inputs and return a result. Kotlin's `fun` keyword begins its declaration. A **parameter** is the input name inside the function; an **argument** is the value supplied by a caller.

```kotlin
fun heading(title: String): String {
    return "Note: $title"
}
```

The first line names the function `heading`. In parentheses, `title: String` declares a text parameter. The colon after the parentheses introduces the **return type**, here `String`. Inside the braces, `return` sends the constructed message back to the caller. It does not print that message. The caller below uses `println` to display the returned value.

Unlike a Java method declaration, the return type follows the parameters. The function can be declared directly in a Kotlin script.

In [ ]:
fun heading(title: String): String {
    return "Note: $title"
}
println(heading("Travel"))

The argument `"Travel"` becomes `title`, and the call returns `"Note: Travel"`. The outer `println` displays it. The function can now be called with another title without copying the template.

### Defaults and Named Arguments

A **default argument** supplies a value when a caller leaves an argument out. A **named argument** identifies the parameter being filled. They solve different problems: defaults reduce repeated choices, while names make a call easier to read.

```kotlin
fun labeledTitle(title: String, prefix: String = "Note"): String {
    return "$prefix: $title"
}
```

Here, `prefix` is optional at the call site because its declaration gives it a default. Supplying `prefix = "Reminder"` replaces that default for this call only. It does not change future calls.

In [ ]:
fun labeledTitle(title: String, prefix: String = "Note"): String {
    return "$prefix: $title"
}
println(labeledTitle("Travel"))
println(labeledTitle(title = "Travel", prefix = "Reminder"))

The lines are `Note: Travel` and `Reminder: Travel`. The first call omits `prefix`; the second names both arguments. A name at a call site must match the parameter name in the declaration.

### One-Expression Functions

An **expression-bodied function** uses `=` followed by the expression that supplies its result. Kotlin can infer its return type from that expression.

```kotlin
fun noteCountLabel(count: Int) = "$count notes"
```

This function returns a `String`. There are no braces and no `return` keyword in this form. It is useful for one clear calculation or message. Use a block body when several steps make the logic easier to read. A block-bodied function returning a value needs an explicit return type; do not assume the expression-body inference rule applies to both forms.

### Make Missing Data Explicit

A **nullable type** permits `null`, a special value meaning no value is present. `String` requires a string; `String?` permits either a string or `null`. The question mark is part of the type, not a new kind of string.

```kotlin
val enteredName: String = "Sam"
val optionalName: String? = null
```

`""` is an existing string with zero characters. It is different from `null`. The literal text `"null"` is also a string, not a missing value.

Before calling a method on nullable text, establish that a string exists. One option is an explicit check. Within the non-null branch below, the compiler recognizes that this stable local value is a `String`. This narrowing is called a **smart cast**. `uppercase()` returns uppercase text; it does not modify the original string.

In [ ]:
val optionalName: String? = "Sam"
if (optionalName != null) {
    println(optionalName.uppercase())
} else {
    println("Guest")
}

This prints `SAM`. With `null`, the `else` branch would print `Guest`. This automatic narrowing works here because the local `val` cannot be reassigned between the check and use. Do not expect every changing property to be narrowed the same way.

### Safe Calls and a Fallback

A **safe call**, written `?.`, calls a method only when the value on its left is not null. Otherwise it skips the call and supplies null. The **Elvis operator**, written `?:`, supplies its right-hand value only when the result on its left is null.

```kotlin
name?.uppercase() ?: fallback
```

Work from left to right. For `"Sam"`, the safe call returns `"SAM"`, so the expression uses that result. For `null`, `uppercase()` is not called; the left side remains null, so the expression uses `fallback`. For `""`, the call returns an empty string, which is not null, so no fallback is used. A fallback for missing data does not also validate blank text.

### Why `!!` Is Usually the Wrong Fix

The **non-null assertion operator**, `!!`, demands a non-null value. If the value is null, it throws a null-pointer exception and execution fails. It does not create a value or perform recovery.

```kotlin
// Intentionally unsafe example; do not run as a normal lesson cell.
val missingName: String? = null
// println(missingName!!.uppercase()) // would fail at runtime
```

When absence is expected, choose an explicit check or a safe call and fallback. Use an assertion only when a separate, reliable guarantee justifies it; adding `!!` merely to silence a compiler error discards useful protection.

In [ ]:
val presentName: String? = "Sam"
val missingName: String? = null
println(presentName?.uppercase())
println(missingName?.uppercase())

The output is `SAM` followed by `null`. The second line is how `println` displays a missing value; the variable does not contain the literal string `"null"`. The safe call prevents the method call on a missing string, but it does not choose a user-facing message.

In [ ]:
val safeLabel: String = missingName?.uppercase() ?: "Guest"
println(safeLabel)

This prints `Guest`. The fallback turns the nullable result into a non-null string. The type annotation on `safeLabel` confirms that both possible paths supply text. The animation below traces the same two-stage decision.

<details class="animation-panel" open>
<summary>Trace present and missing names — show or hide animation</summary>
<p><img src="media/02_functions_and_null_safety/nullable_fallback.gif" alt="Present Sam becomes SAM and skips the fallback. Missing null skips uppercase and selects Visitor." width="960" style="max-width:100%;height:auto;"></p>
</details>

The first path has a string to transform. The second has no string, so it selects the caller’s fallback. Neither path changes the original input. The 10-second animation loops; closing its panel hides the motion.

[View the labeled still diagram](media/02_functions_and_null_safety/nullable_fallback_still.png).

## Worked Example

### A Safe Display Name

Our app needs a display label for an optional name. A present name should appear in uppercase. A missing name should use `Guest`, unless the caller supplies another fallback. A separate function formats a note count.

1. Declare `displayName` with `name: String?` so the type admits missing input.
2. Give `fallback` a non-null `String` type and a default of `"Guest"`. That ensures the fallback can supply real text.
3. Return the safe-call result or the fallback. Both successful paths produce text, so the function returns `String`, not `String?`.
4. Declare `noteCountLabel` with an expression body. Its template supplies a string directly.
5. Call the functions with present and missing data. Named arguments make the missing-name choice clear.

The complete example joins these small decisions. It creates no Android screen and makes no network request; it checks the formatting rule itself.

In [ ]:
fun displayName(name: String?, fallback: String = "Guest"): String {
    return name?.uppercase() ?: fallback
}
fun noteCountLabel(count: Int) = "$count notes"
println(displayName(name = "Sam"))
println(displayName(name = null, fallback = "Visitor"))
println(noteCountLabel(3))

The output is:

```text
SAM
Visitor
3 notes
```

The first call has a name, so it ignores the default fallback. The second supplies null and overrides the default with `Visitor`. The last call inserts the integer into a string. Notice that the default is used only when the argument is omitted; explicitly passing a different string selects that string.

An empty name would produce a blank output line, because this contract handles missing data only. Deciding that empty or whitespace-only names are invalid is a separate rule that must be stated and tested.

## Guided Practice

### Trace missing, empty, and present text

Before running the next cell, predict its three output lines. The brackets make an empty result visible. Identify which call uses its fallback and explain why the empty-title call does or does not use `Draft`. Then run the cell to check your prediction.

In [ ]:
Your prediction:
Three output lines:
Which call uses the fallback, and why:


In [ ]:
fun previewTitle(title: String?, fallback: String = "Untitled"): String =
    title?.uppercase() ?: fallback
val missingPreview = previewTitle(title = null)
val emptyPreview = previewTitle(title = "", fallback = "Draft")
val presentPreview = previewTitle(title = "App sketch")
println("[$missingPreview]")
println("[$emptyPreview]")
println("[$presentPreview]")

<details>
<summary>Show answer</summary>

```text
[Untitled]
[]
[APP SKETCH]
```

For `null`, the safe call skips `uppercase()` and produces `null`, so Elvis selects the default fallback `Untitled`. Empty text is a present string. Its uppercase result is still empty, so Elvis does not select `Draft`. The third call returns uppercase text. `?:` checks for null, not whether the text has any characters. The brackets belong to the printed test display, not the function result.

</details>

### Complete a reusable heading

The starter is runnable but unfinished: it always returns `fallback`. Replace that expression with a safe call and Elvis fallback so present titles become uppercase and missing titles use the fallback. Keep the expression-body form.

Add three calls and print their returned values: `"Read notes"`; `null` with the default fallback; and `null` with the named fallback `"Local draft"`. Then change only the last call's fallback to `"Unsaved"` and run again. The first two results should stay the same.

In [ ]:
fun editorHeading(title: String?, fallback: String = "Draft"): String =
    fallback // TODO: Replace this expression with safe uppercase and fallback logic.
// TODO: Add and print the three calls described above.

<details>
<summary>Show answer</summary>

```kotlin
fun editorHeading(title: String?, fallback: String = "Draft"): String =
    title?.uppercase() ?: fallback
println(editorHeading("Read notes"))
println(editorHeading(null))
println(editorHeading(title = null, fallback = "Local draft"))
```

The result is:

```text
READ NOTES
Draft
Local draft
```

The function's `=` introduces its returned expression. The default argument applies when the caller leaves out `fallback`. A named argument supplies the caller's replacement. Changing only the last call to `fallback = "Unsaved"` changes only the last output line to `Unsaved`.

A safe call alone returns a nullable result, which does not satisfy the declared `String` return type. Elvis supplies a non-null string for the missing case. Avoid returning the literal word `fallback` in quotes; use the parameter's value.

</details>

### Repair an unsafe assumption

This **intentionally faulty snippet** is shown as text so it does not stop the other exercises:

```kotlin
fun headingText(title: String?): String = title!!.uppercase()
println(headingText(null))
```

Explain what `!!` claims and why this call fails. Then write a corrected `headingText` function that uppercases present text and returns `"Untitled"` for null, without `!!`. Test it with `"Lab ideas"` and `null`.

In [ ]:
Your diagnosis:
What the assertion claims:
Why the null call fails:


In [ ]:
// TODO: Write the corrected headingText function and print both test results.

<details>
<summary>Show answer</summary>

```kotlin
fun headingText(title: String?): String = title?.uppercase() ?: "Untitled"
println(headingText("Lab ideas"))
println(headingText(null))
```

`!!` asserts that the value is not null; it does not supply a replacement. Here it throws before `uppercase()` or `println` can finish. The safe call skips the method for null, and Elvis provides the required fallback.

```text
LAB IDEAS
Untitled
```

Changing the parameter to `String` would reject null callers instead of meeting this task's requirement to handle missing text.

</details>

## Independent Practice

### Format a note title for two screens

A note preview needs a reusable `formatNoteTitle` function. It accepts `title: String?` and `fallback: String = "Untitled"`, and returns a `String`.

Use these rules:

- For present, nonempty text, return `Note: ` followed by its uppercase form.
- For null, return `Note: ` followed by the caller's fallback, keeping the fallback's capitalization.
- For empty text (`""`), return `Add a title`. This is a separate editing cue, even when a fallback was supplied.

Use a block body with a returned result. Combine the safe call and Elvis operator with the `if` and string-template ideas from Lesson 1. Do not use `!!`. Keep the same function for all tests:

| Title input | Fallback argument | Expected returned string |
| --- | --- | --- |
| `"Lab ideas"` | Omitted | `Note: LAB IDEAS` |
| `null` | Omitted | `Note: Untitled` |
| `null` | `"Draft"` | `Note: Draft` |
| `""` | Omitted | `Add a title` |
| `""` | `"Draft"` | `Add a title` |

Write the function and calls in the next cell, and print each returned string. Treat this as a text-formatting rule; no Android interface is required.

In [ ]:
// TODO: Write formatNoteTitle with the required parameters and String result.
// TODO: Print the results for all five input combinations in the table.

<details>
<summary>Show answer</summary>

One solution is:

```kotlin
fun formatNoteTitle(title: String?, fallback: String = "Untitled"): String {
    val heading = title?.uppercase() ?: fallback
    return if (title == "") "Add a title" else "Note: $heading"
}
println(formatNoteTitle("Lab ideas"))
println(formatNoteTitle(null))
println(formatNoteTitle(title = null, fallback = "Draft"))
println(formatNoteTitle(""))
println(formatNoteTitle(title = "", fallback = "Draft"))
```

The safe call uppercases only a present title. Elvis supplies the fallback only for null; it leaves the fallback's own capitalization unchanged. The separate `title == ""` test provides the editing cue. Without that test, an empty title would produce `Note: `.

The block body uses `return` to send the finished string to its caller. The caller prints the result, which keeps the formatting rule reusable by either screen. Each printed line matches the table above. Applying `uppercase()` after selecting the fallback would also uppercase `Draft`, violating this task's rule.

</details>

### Explain what your tests establish

After running the tests, compare the two calls that supply `Draft`: one receives null and the other receives empty text. State their actual results and explain why they differ. Which test also checks that the fallback's capitalization is preserved?

In [ ]:
Your test explanation:
Null with Draft:
Empty text with Draft:
Why the results differ and which test checks capitalization:


<details>
<summary>Show answer</summary>

Null with `Draft` returns `Note: Draft`; empty text with `Draft` returns `Add a title`. Null means no string is present, so the safe call yields null and Elvis supplies `Draft`. Empty text is a string, so Elvis does not use the fallback; the explicit empty-text check supplies the editing cue. The null-with-`Draft` test checks capitalization too: `Note: DRAFT` would reveal that uppercase conversion was applied to the fallback. Successful present-text output alone would miss these differences.

</details>

## Summary

- `fun` declares a function. Parameters describe inputs; `return` sends a result to the caller.
- Defaults apply when arguments are omitted. Named arguments make a call's choices explicit.
- An expression body returns its expression and can infer its result type.
- `String?` admits null; `String` does not. Empty text is still a string.
- `?.` skips access on null. `?:` supplies a fallback for null. Neither rule rejects empty text.
- `!!` can fail at runtime. Prefer explicit handling when absence is expected.

In the next lesson, we will group related values into classes and give those objects useful operations. Functions and nullable types will remain part of their contracts.

## Reflection

Without looking back, state what a safe call does when its receiver is null. Then choose another app field where missing text and empty text could deserve different messages. Briefly describe the two messages and the rule you would put in a reusable function. No code is required.

In [ ]:
Your reflection:
Safe-call behavior for null:
App field and separate missing/empty rules:


<details>
<summary>Show answer</summary>

A safe call skips the method or property access for a null receiver and produces null. For example, a profile's nickname could use `Profile not loaded` when the value is missing, but `Choose a nickname` when the user has left the text empty. A function could return the appropriate display text from that input. Your app may use other messages; make the distinction an explicit rule rather than assuming Elvis treats empty text as missing.

</details>

## Supplemental Reading

- [Kotlin functions](https://kotlinlang.org/docs/functions.html) — declarations, return types, default and named arguments, and expression bodies.
- [Kotlin null safety](https://kotlinlang.org/docs/null-safety.html) — nullable types, checks, safe calls, Elvis, and non-null assertions.